This notebook covers the advanced EC2 features that architects use to optimise performance, availability, and launch speed: the full AMI lifecycle, placement groups for controlling physical placement of instances, Hibernate, the Nitro platform, and a few other exam-relevant features.

## AMI Lifecycle

An AMI progresses through a clear lifecycle:

```
Running Instance
      │
      ▼  CreateImage (or create from snapshot)
    AMI  ──── copy ────► AMI in another region
      │
      ▼  RunInstances
  New Instance
      │
      ▼  DeregisterImage (when no longer needed)
   (deleted)
```

### Creating an AMI
When you call `CreateImage` on a running or stopped instance, AWS:
1. Optionally reboots the instance (for filesystem consistency)
2. Takes a snapshot of each attached EBS volume
3. Registers an AMI that points to those snapshots

The AMI itself is just metadata — the actual data lives in the EBS snapshots stored in S3 (managed by AWS, not visible in your S3 buckets).

### AMI sharing and permissions

| Visibility | Description |
|---|---|
| **Private** (default) | Only your account can use it |
| **Shared with specific accounts** | Grant specific AWS account IDs access |
| **Public** | Any AWS account can launch it |

To share an encrypted AMI, you must also share the underlying KMS key with the target account.

### Copying AMIs across regions
AMIs are region-specific. `CopyImage` creates a new AMI (with new AMI ID and new snapshots) in the target region. Use cases:
- Multi-region deployments with consistent images
- Disaster recovery — keep a copy of your golden AMI in a second region

### Deregistering an AMI
Deregistering removes the AMI registration but does **not** delete the underlying snapshots. You must delete the snapshots separately to stop incurring storage costs.

## AMI Baking Strategies

There is a spectrum between fully bootstrapped instances and fully baked AMIs:

```
Fully Bootstrapped          Hybrid (most common)          Fully Baked
       │                            │                           │
Base AMI +              Base AMI + dependencies        App + all config
long User Data          baked in + app pulled            in the AMI
script                  at launch
       │                            │                           │
Slowest launch          Balanced                         Fastest launch
Most flexible           Practical default                Least flexible
```

### EC2 Image Builder
**EC2 Image Builder** automates the creation, testing, and distribution of AMIs on a schedule or when a source image is updated.

Pipeline stages:
1. **Source** — base AMI (e.g. Amazon Linux 2023)
2. **Build components** — install packages, run scripts, apply config
3. **Test components** — run automated tests against the built image
4. **Distribute** — copy to target regions, share with accounts

Image Builder is the AWS-native solution for maintaining a patched, tested golden AMI across your organization.

## Placement Groups

By default, AWS distributes instances across underlying hardware to minimise correlated failures. **Placement groups** let you override this behaviour — either packing instances tightly together for performance, or spreading them further apart for resilience.

There are three types:

---

### 1. Cluster Placement Group

All instances are packed into **one AZ on the same physical rack** (or as close to it as possible).

```
AZ: us-east-1a
  └── Single rack
        ├── instance-1
        ├── instance-2
        └── instance-3
```

| | Cluster |
|---|---|
| **Network** | Up to 10 Gbps between instances (vs 5 Gbps normally); lowest latency |
| **Risk** | If the rack fails, all instances fail |
| **AZs** | Single AZ only |
| **Use case** | HPC, tightly coupled parallel jobs, big data that needs fast node-to-node communication |

---

### 2. Spread Placement Group

Each instance is placed on a **distinct underlying hardware rack** — separate power and networking per instance.

```
AZ: us-east-1a        AZ: us-east-1b        AZ: us-east-1c
  Rack 1                Rack 2                Rack 3
  instance-1            instance-2            instance-3
```

| | Spread |
|---|---|
| **Risk** | Isolated hardware failure — one instance at a time |
| **AZs** | Spans multiple AZs |
| **Limit** | Max **7 instances per AZ** per placement group |
| **Use case** | Small number of critical instances that must not fail simultaneously — primary/standby databases, ZooKeeper nodes, HA control plane components |

---

### 3. Partition Placement Group

Instances are divided into **partitions** (logical segments). Each partition maps to a distinct set of racks — racks in one partition share no hardware with racks in another.

```
AZ: us-east-1a
  Partition 1 (Rack A, B)      Partition 2 (Rack C, D)     Partition 3 (Rack E, F)
  instance-1                   instance-4                   instance-7
  instance-2                   instance-5                   instance-8
  instance-3                   instance-6                   instance-9
```

| | Partition |
|---|---|
| **Risk** | A rack failure affects one partition, not others |
| **AZs** | Can span multiple AZs |
| **Scale** | Up to 7 partitions per AZ; hundreds of instances total |
| **Metadata** | Instances can query which partition they are in via IMDS |
| **Use case** | Distributed systems that use rack-awareness — HDFS, HBase, Cassandra, Kafka |

---

### Choosing a placement group

| Need | Use |
|---|---|
| Lowest latency between instances | Cluster |
| Maximum isolation, small critical fleet | Spread |
| Large distributed system with rack-awareness | Partition |

## EC2 Hibernate

**Hibernate** saves the in-memory (RAM) state of an instance to the root EBS volume, then stops the instance. On restart, the RAM is restored and the instance resumes exactly where it left off — OS boot is skipped entirely.

```
Normal Stop:   RAM is lost  →  full OS boot on restart
Hibernate:     RAM saved    →  resume from saved state (seconds)
```

### Requirements
- Root volume must be EBS-encrypted (RAM contents are sensitive)
- Root volume must be large enough to hold the RAM dump
- Instance RAM must be less than 150 GB
- Not supported on bare metal instances
- Max hibernate duration: **60 days**

### Use cases
- Long-running processes that take time to warm up (caches, in-memory state)
- Services with slow startup — hibernate overnight instead of stopping
- Save the cost of running an instance continuously while preserving application state

## The AWS Nitro System

The **Nitro System** is the underlying platform for most modern EC2 instances (5th generation onwards). It offloads virtualisation functions — networking, storage, security — to dedicated Nitro cards and a lightweight Nitro hypervisor, freeing almost all of the host's CPU and memory for the customer instance.

### What Nitro enables

| Capability | Detail |
|---|---|
| **Near bare-metal performance** | Minimal hypervisor overhead |
| **Bare metal instances** | No hypervisor at all — direct hardware access (e.g. `i3.metal`, `m7i.metal`) |
| **EBS encryption** | Performed by the Nitro card at hardware speed — no CPU overhead |
| **Higher network bandwidth** | Up to 200 Gbps on some instances |
| **Enhanced security** | Hypervisor and host are immutable; no AWS operator access to guest memory |

### Bare metal instances
Bare metal instances give you direct access to the underlying server's CPU and memory. Use cases:
- Workloads that need access to hardware features not exposed through virtualisation
- Running your own hypervisor (e.g. VMware Cloud on AWS)
- Licensing that requires physical CPU socket access

## Other Exam-Relevant Features

### EC2 Instance Connect
Browser-based or CLI SSH access to Linux instances without managing SSH keys. AWS pushes a temporary one-time-use public key to the instance immediately before connection. Requires the instance to have the EC2 Instance Connect package installed and appropriate security group rules.

### EC2 Serial Console
Provides access to the instance's serial port — useful for troubleshooting instances that are unresponsive over SSH (e.g. after a bad kernel update or network misconfiguration). Available on Nitro-based instances.

### Tenancy options recap

| Tenancy | Hardware sharing | Visibility into host | BYOL |
|---|---|---|---|
| **Shared** (default) | Shared with other customers | No | No |
| **Dedicated Instance** | Your instances only | No | No |
| **Dedicated Host** | Your instances only | Yes (socket/core level) | Yes |
| **Bare Metal** | No hypervisor | Direct hardware access | Yes |

### Instance stop vs terminate vs hibernate

| Action | RAM | EBS root | Instance store | Billing |
|---|---|---|---|---|
| **Stop** | Lost | Retained | Lost | Stops |
| **Hibernate** | Saved to EBS | Retained | Lost | Stops |
| **Terminate** | Lost | Deleted (by default) | Lost | Stops |

## Working with AMIs and Placement Groups via boto3

In [ ]:
import boto3

ec2 = boto3.client('ec2', region_name='us-east-1')

In [ ]:
# Create an AMI from a running or stopped instance
response = ec2.create_image(
    InstanceId='i-0abcdef1234567890',   # replace with your instance ID
    Name='my-app-golden-ami-v1',
    Description='Baked AMI with app v1 installed',
    NoReboot=False                       # False = reboot for filesystem consistency
)
print(f"AMI ID: {response['ImageId']}")

In [ ]:
# Copy an AMI to another region for DR
dest_ec2 = boto3.client('ec2', region_name='eu-west-1')
response = dest_ec2.copy_image(
    SourceImageId='ami-0abcdef1234567890',  # AMI in us-east-1
    SourceRegion='us-east-1',
    Name='my-app-golden-ami-v1-dr-copy'
)
print(f"Copied AMI ID in eu-west-1: {response['ImageId']}")

In [ ]:
# Create placement groups of each type
for strategy in ['cluster', 'spread', 'partition']:
    kwargs = {'GroupName': f'demo-{strategy}', 'Strategy': strategy}
    if strategy == 'partition':
        kwargs['PartitionCount'] = 3
    ec2.create_placement_group(**kwargs)
    print(f"Created {strategy} placement group")

In [ ]:
# Launch an instance into a cluster placement group
response = ec2.run_instances(
    ImageId='ami-0c02fb55956c7d316',
    InstanceType='c5n.large',           # network-optimised — good for cluster PG
    MinCount=1,
    MaxCount=1,
    Placement={
        'GroupName': 'demo-cluster',
        'AvailabilityZone': 'us-east-1a'
    }
)
print(f"Launched: {response['Instances'][0]['InstanceId']}")

In [ ]:
# List your own AMIs
response = ec2.describe_images(Owners=['self'])
for image in sorted(response['Images'], key=lambda x: x['CreationDate'], reverse=True):
    print(f"{image['ImageId']}  {image['CreationDate'][:10]}  {image['Name']}")

## Summary

| Concept | Key Takeaway |
|---|---|
| AMI | Region-specific; backed by EBS snapshots; deregister does not delete snapshots |
| AMI sharing | Share with specific accounts or make public; encrypted AMIs require sharing the KMS key |
| EC2 Image Builder | Automates AMI creation, testing, and distribution on a pipeline |
| Cluster PG | Same rack; lowest latency, highest throughput; single AZ; risk of correlated failure |
| Spread PG | One instance per rack; max 7 per AZ; maximum isolation for critical instances |
| Partition PG | Groups of instances per partition; partitions isolated from each other; for rack-aware distributed systems |
| Hibernate | Saves RAM to encrypted EBS; instance resumes without full OS boot; max 60 days |
| Nitro System | Modern hypervisor platform; near bare-metal performance; hardware-accelerated EBS encryption |
| Bare metal | No hypervisor; direct hardware access; needed for VMware or hardware-licensed software |